# Gemma Kavach - LoRA Fine-Tuning on Kaggle

This notebook fine-tunes Gemma 4 E2B with LoRA for emergency classification + crowd reasoning.

**Required Kaggle Settings:**
- Accelerator: GPU T4 x2 (or P100)
- Internet: ON (to download model)
- Environment: Latest Python + PyTorch

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2: Import Libraries

In [ ]:
import os
import torch
import json
import random
from unsloth import FastModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 3: Define Training Data

This is your emergency classification + crowd reasoning dataset.

In [ ]:
# Seed data with all categories
SEED_DATA = [
    # Emergency categories
    {
        "text": "bache ko uski mummy nahi mil rahi hai, wo food court ke paas hai.",
        "category": "child_lost",
        "severity": "CRITICAL",
        "reasoning": "A child is separated from their parents in a crowded area. High risk of abduction or harm.",
        "action": "Dispatch security to food court immediately. Make PA announcement."
    },
    {
        "text": "People are pushing and someone fell down near gate 3! Bhagdad mach gai hai!",
        "category": "crowd_panic",
        "severity": "CRITICAL",
        "reasoning": "Crowd crushing and trampling risk at gate 3. High potential for mass casualties.",
        "action": "Deploy rapid response team to gate 3, open emergency exits, issue calming announcements."
    },
    {
        "text": "There's a small trash can fire near the restrooms. dhua nikal raha hai.",
        "category": "small_fire",
        "severity": "HIGH",
        "reasoning": "Fire can spread quickly in crowded areas causing panic and burns.",
        "action": "Dispatch fire wardens with extinguishers, prepare for localized evacuation."
    },
    {
        "text": "My grandmother fainted in the heat. ek aurat behosh ho gai.",
        "category": "medical_help",
        "severity": "HIGH",
        "reasoning": "Medical emergency (syncope) likely due to heat exhaustion. Requires immediate care.",
        "action": "Send paramedics to location with stretcher and hydration."
    },
    {
        "text": "I lost my blue backpack near the entry.",
        "category": "lost_item",
        "severity": "LOW",
        "reasoning": "Missing personal property. No immediate threat to life or safety.",
        "action": "Direct user to lost and found counter."
    },
    {
        "text": "mujhe kuch samajh nahi aa raha, yahan koi tamil bolne wala hai?",
        "category": "need_interpreter",
        "severity": "MEDIUM",
        "reasoning": "Language barrier causing confusion. Needs translation assistance to navigate or report issues.",
        "action": "Dispatch multilingual volunteer or connect to phone translation service."
    },
    
    # Crowd safety reasoning
    {
        "text": "Crowd analysis detected High density and Chaotic motion, leading to a CRITICAL risk level. Explain why this is dangerous and what actions should be taken.",
        "category": "crowd_safety_reasoning",
        "severity": "CRITICAL",
        "reasoning": "High density combined with chaotic motion creates immediate trampling risk. People cannot control their movement in dense crowds, and panic behavior amplifies danger. Mass casualties possible within seconds.",
        "action": "Deploy rapid response team immediately. Open emergency exits. Issue calming announcements. Establish crowd flow control barriers."
    },
    {
        "text": "Crowd analysis detected High density and Calm motion, leading to a MODERATE risk level. Explain why this is dangerous and what actions should be taken.",
        "category": "crowd_safety_reasoning",
        "severity": "MODERATE",
        "reasoning": "High density alone is concerning even without panic. Crowd crush can occur from simple movement like people trying to see something. Requires monitoring to prevent escalation.",
        "action": "Monitor closely. Prepare crowd dispersal plan. Station personnel at choke points. Limit further ingress."
    },
    {
        "text": "Crowd analysis detected Medium density and Chaotic motion, leading to a HIGH risk level. Explain why this is dangerous and what actions should be taken.",
        "category": "crowd_safety_reasoning",
        "severity": "HIGH",
        "reasoning": "Chaotic motion in medium-density areas suggests localized panic or conflict. May spread rapidly if not contained. Could escalate to mass panic event.",
        "action": "Investigate source of chaos immediately. Deploy security to affected zone. Prevent crowd from entering area. Prepare medical response."
    },
    {
        "text": "Crowd analysis detected Low density and Chaotic motion, leading to a MODERATE risk level. Explain why this is dangerous and what actions should be taken.",
        "category": "crowd_safety_reasoning",
        "severity": "MODERATE",
        "reasoning": "Chaotic motion in sparse crowds typically indicates specific incident like fight, medical emergency, or fire. Not mass panic, but requires immediate attention.",
        "action": "Dispatch investigation team. Check for medical emergencies or security incidents. Monitor for crowd gathering around incident."
    },
]

print(f"✅ Loaded {len(SEED_DATA)} seed examples")
print(f"Categories: {set([x['category'] for x in SEED_DATA])}")

## Step 4: Generate Synthetic Dataset

In [ ]:
def generate_variations(seeds, count=10000):
    """Generate variations by appending locations to emergency text"""
    print(f"Generating {count} synthetic variations...")
    variations = []
    locations = ["gate 1", "gate 2", "main stage", "food court", "exit A", "exit B", 
                 "parking lot", "restroom area", "vip section", "zone A", "zone B", "zone C"]
    
    for i in range(count):
        base = random.choice(seeds)
        loc = random.choice(locations)
        
        new_text = base["text"]
        # Only modify emergency categories, keep crowd reasoning as-is
        if base["category"] in ["child_lost", "crowd_panic", "small_fire", "medical_help", "lost_item", "need_interpreter"]:
            if "near" in base["text"] or "paas" in base["text"]:
                new_text = base["text"] + f" around {loc}"
            else:
                new_text = base["text"].replace(".", f" near {loc}.")
        
        variations.append({
            "text": new_text,
            "category": base["category"],
            "severity": base["severity"],
            "reasoning": base["reasoning"],
            "action": base["action"]
        })
    
    return variations

# Generate dataset
raw_data = generate_variations(SEED_DATA, count=10000)

# Add seeds multiple times to ensure they're well-learned
raw_data.extend(SEED_DATA * 50)
random.shuffle(raw_data)

print(f"✅ Total dataset size: {len(raw_data)} examples")

## Step 5: Format for Training

In [ ]:
def format_prompt(example):
    """Format examples for Gemma 4 E2B instruction tuning"""
    categories_str = "child_lost, crowd_panic, lost_item, medical_help, need_interpreter, small_fire, crowd_safety_reasoning"
    
    # Crowd reasoning already has the full question
    if example['category'] == "crowd_safety_reasoning":
        prompt = example['text']
    else:
        prompt = f"Classify this emergency into one of these categories: {categories_str}\n\nEmergency: {example['text']}\n\nCategory:"
    
    # Structured JSON output
    target = json.dumps({
        "category": example['category'],
        "severity": example['severity'],
        "reasoning": example['reasoning'],
        "action": example['action']
    })
    
    return {"text": prompt, "label": target}

# Create HuggingFace dataset
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_prompt)

print(f"✅ Dataset formatted: {len(dataset)} examples")
print("\n📋 Sample formatted example:")
print("PROMPT:", dataset[0]['text'][:200], "...")
print("TARGET:", dataset[0]['label'][:200], "...")

## Step 6: Load Model and Tokenizer

In [ ]:
print("🚀 Loading Gemma 4 E2B model...")

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    load_in_4bit=True,  # Memory efficient
)

print(f"✅ Model loaded: {type(model).__name__}")
print(f"✅ Tokenizer loaded: {type(tokenizer).__name__}")

## Step 7: Add LoRA Adapter

In [ ]:
print("🔧 Configuring LoRA adapter...")

model = FastModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA adapter configured")
model.print_trainable_parameters()

## Step 8: Format Dataset for SFTTrainer

In [ ]:
def formatting_prompts_func(examples):
    """Apply chat template for instruction tuning"""
    texts = []
    for text, label in zip(examples["text"], examples["label"]):
        messages = [
            {"role": "user", "content": text},
            {"role": "model", "content": label}
        ]
        chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(chat_text)
    return {"formatted_text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Chat template applied to {len(formatted_dataset)} examples")
print("\n📋 Sample formatted chat:")
print(formatted_dataset[0]['formatted_text'][:300], "...")

## Step 9: Initialize Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="formatted_text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=1000,  # Adjust based on time available
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=250,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("✅ Trainer initialized and ready!")

## Step 10: Train! 🚀

**Expected time:** 2-4 hours on T4 x2

**Watch for:**
- Loss should decrease over time (target: <1.0)
- If loss is very high initially (13-15), that's normal for multimodal Gemma - it will decrease
- Checkpoints saved every 250 steps

In [ ]:
print("🧠 Starting training...\n")
print("⏱️ Estimated time: 2-4 hours")
print("💾 Checkpoints will be saved every 250 steps\n")

trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

## Step 11: Save the Adapter

In [ ]:
output_dir = "gemma_kavach_lora_adapter"

print(f"💾 Saving LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("\n✅ Adapter saved successfully!")
print(f"📁 Files in {output_dir}:")
!ls -lh {output_dir}

## Step 12: Test the Fine-Tuned Model

In [ ]:
print("🧪 Testing the fine-tuned model...\n")

test_prompts = [
    "bacha kho gaya hai zone C mein",
    "bheed mein panic ho gaya hai",
    "Crowd analysis detected High density and Chaotic motion, leading to a CRITICAL risk level. Explain why this is dangerous and what actions should be taken.",
]

for test_text in test_prompts:
    messages = [{"role": "user", "content": test_text}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"INPUT: {test_text}")
    print(f"\nOUTPUT:\n{response}")
    print(f"{'='*60}")

## Step 13: Download the Adapter

**Important:** Download these files to deploy on your local machine:
- `adapter_config.json`
- `adapter_model.safetensors` (or `adapter_model.bin`)

Place them in `Gemma_ke_balak/FineTunedModel/` on your local machine.

In [ ]:
print("📦 Creating download archive...")
!zip -r gemma_kavach_adapter.zip {output_dir}

print("\n✅ Archive created: gemma_kavach_adapter.zip")
print("📥 Download this file from Kaggle Output tab")
print("\nℹ️ To deploy:")
print("1. Download gemma_kavach_adapter.zip")
print("2. Extract to Gemma_ke_balak/FineTunedModel/")
print("3. Restart GemmaServer")
print("4. Verify adapter loads successfully")